# TabICL: will sleep start in the next 30 or 60 minutes?

This walkthrough adds [jingang/TabICL](https://huggingface.co/jingang/TabICL) to our existing LightGBM, XGBoost, random forest, Jev and Kumo experiments. The SQL labels are **sleep onset within the window**, despite their `is_asleep_*` names. A short nap that starts and finishes inside the window is still a positive example.

TabICL is a pretrained tabular transformer. `fit(X_context, y_context)` supplies labelled examples; it does not train the neural weights. `predict_proba(X_query)` performs in-context inference. Each horizon gets its own classifier. We use the v2 classifier checkpoint, pinned to a Hub commit, and one ensemble member as a laptop starting point (the library default is larger). All baby data stays local for TabICL.

[Official usage and API](https://github.com/soda-inria/tabicl). Our implementation is `ml/baby_ml/tabicl.py`.

## 1. Run the local validation baseline in VS Code

Open this `.ipynb` file in VS Code on the Mac Studio. Use **Select Kernel → Python Environments** and choose the repo's `.venv/bin/python`. If the environment is missing the notebook dependencies, run `uv sync --group ml --group tabicl` from the repo root.

Run the setup cell, then **Load dev data**. That cell reads `baby_data/profiles.yml` and queries `ml.ml_sleep_training_set` through the local PostgreSQL Unix socket. It saves a frozen snapshot under `ml/data/dev/`. Set `SNAPSHOT` to a previously printed snapshot path to run offline.

On a cache miss the notebook runs TabICL in a separate Python process to keep PyTorch and the tree models’ OpenMP runtimes apart. The first inference downloads the pinned weights if needed. Full-context CPU inference can take time. On macOS, install `libomp` with Homebrew if LightGBM reports a missing `libomp.dylib`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from baby_ml import Comparison, FeatureSpec
from baby_ml import bootstrap_by_day, load_training_set, metrics_table, plot_performance
from baby_ml.data import SNAPSHOT_DIR, pull_snapshot
from baby_ml.settings import DatabaseSettings
from baby_ml.tabicl import LABELS, TabICLModel, TabICLSpec

SNAPSHOT = None  # Set to Path('/absolute/path/to/dev/snapshot.parquet') to replay.
RUN_TREE_COMPARISON = True
RUN_TEST_COMPARISON = False

SPLIT = 'split_forward_time'
spec = TabICLSpec(n_estimators=1, device='cpu')
tabicl = TabICLModel(spec)
recent = TabICLModel(spec.model_copy(update={'context_splits': ('train', 'val')}))

### Load dev data

In [ ]:
snapshot = SNAPSHOT
if snapshot is None:
    snapshot = pull_snapshot(DatabaseSettings.local_dev(), directory=SNAPSHOT_DIR / 'dev')
df = load_training_set(path=snapshot)
print(snapshot, df.shape)
display(df.groupby(SPLIT)[list(LABELS)].agg(['size', 'mean']))

## 2. Check features and time splits

All models use the same frozen frame and row order. `FeatureSpec` excludes identifiers, split columns, both outcomes and `minutes_to_next_sleep`. TabICL fits preprocessing using the labelled context. Validation uses training rows as context.

In [ ]:
features = spec.features.feature_columns(df)
assert not set(features) & set(FeatureSpec.LABELS)
assert (df[LABELS[0]] <= df[LABELS[1]]).all()
display(pd.DataFrame({'feature': features, 'dtype': [str(df[c].dtype) for c in features]}))
display(df.groupby([SPLIT, 'baby_name'], observed=True).agg(
    rows=('prediction_id', 'size'), start=('prediction_time', 'min'), end=('prediction_time', 'max')))

def baby_days(split):
    rows = df.loc[df[SPLIT] == split]
    return (rows.baby_id.astype(str) + '|' + pd.to_datetime(rows.calendar_date).dt.strftime('%Y-%m-%d')).to_numpy()

## 3. Score the pretrained TabICL baseline

TabICL uses labelled training rows as in-context examples with frozen pretrained weights. The validation rows are held out. The constant-probability baseline uses each outcome's training prevalence. Both baselines are scored on the same validation rows.

Restart the kernel before running the notebook if PyTorch was imported in an earlier session.

In [ ]:
def score_tabicl(model, split):
    try:
        return list(model.predict(df, split, cache_only=True).values())
    except FileNotFoundError:
        pass

    command = [
        sys.executable, '-m', 'baby_ml.tabicl', split,
        '--snapshot', str(snapshot.resolve()),
        '--context', *model.spec.context_splits,
        '--estimators', str(model.spec.n_estimators),
        '--device', model.spec.device,
    ]
    print(f'Computing {model.name} predictions for {split} in a separate process.', flush=True)
    subprocess.run(command, check=True, env={**os.environ, 'OMP_NUM_THREADS': '1'})
    return list(model.predict(df, split, cache_only=True).values())


tabicl_scores = score_tabicl(tabicl, 'val')
train_rates = df.loc[df[SPLIT] == 'train', list(LABELS)].mean()
prevalence_scores = [
    pred.model_copy(update={
        'name': 'train_prevalence',
        'p': np.full(len(pred.y_true), train_rates[pred.label]),
    })
    for pred in tabicl_scores
]
baseline_scores = tabicl_scores + prevalence_scores
display(metrics_table(baseline_scores).round(3))
for label in LABELS:
    plot_performance([pred for pred in baseline_scores if pred.label == label], title=label)

## 4. Validation comparison with the tree models

The setup enables LightGBM, XGBoost and random forest scoring on the same validation rows. Historical Jev and Kumo caches belong to their original snapshots.

In [ ]:
if RUN_TREE_COMPARISON:
    tree_params = {
        'lightgbm': {'n_jobs': 1},
        'xgboost': {'n_jobs': 1},
        'random_forest': {'n_jobs': 1},
    }
    val = Comparison.run(df, split='val', params=tree_params)
    val = val.with_predictions(baseline_scores)
    display(val.metrics().round(3))

## 5. Ranking, probabilities and uncertainty

PR-AUC (average precision here) measures ranking of sleep-onset examples; higher is better, with the positive base rate as a reference. ROC-AUC also measures ranking, with 0.5 as chance. Brier measures probability error, so lower is better. `mean_predicted` versus `base_rate` flags broad probability bias; it is not a complete calibration test. The base-rate Brier reference uses the observed evaluation prevalence as a descriptive benchmark.

Cumulative recall asks how many positive rows are captured in the highest-scored fraction of rows. These are prediction opportunities, not counts of unique naps. Confidence intervals resample whole baby-days, since adjacent rows are correlated. Gaps are model minus XGBoost: positive favours a model for AUC, negative for Brier. An interval spanning zero does not prove equivalence.

In [ ]:
if RUN_TREE_COMPARISON:
    val.plot_performance();
    display(val.recall_at_percentiles().round(3))
    display(bootstrap_by_day(list(val.predictions.values()), baby_days('val'), reference='xgboost', n_boot=1000).round(3))

## 6. Optional test comparison with both context choices

Set `RUN_TEST_COMPARISON = True` in the setup cell to enable the remaining test sections.

Freeze settings before executing this section. The test set has already been examined in notebooks 04 and 05, so this is an **exploratory comparison**, not a fresh unbiased final evaluation. Reserve new future dates before making deployment claims.

Train-only TabICL matches train-only XGBoost. TabICL with train + validation matches XGBoost refitted on train + validation. This separates algorithm differences from access to newer routines.

In [ ]:
if RUN_TEST_COMPARISON:
    test = df.loc[df[SPLIT] == 'test']
    comparison = Comparison.run(df, kinds=['xgboost'], split='test', params={'xgboost': {'n_jobs': 1}})
    refit_df = df.copy()
    refit_df[SPLIT] = refit_df[SPLIT].replace({'val': 'train'})
    refit = Comparison.run(refit_df, kinds=['xgboost'], split='test', params={'xgboost': {'n_jobs': 1}})
    extras = [p.model_copy(update={'name': 'xgboost_train_val'}) for p in refit.predictions.values()]
    extras += score_tabicl(tabicl, 'test')
    extras += score_tabicl(recent, 'test')
    comparison = comparison.with_predictions(extras)
    display(comparison.metrics().round(3))
    display(comparison.recall_at_percentiles().round(3))

### Paired comparisons and curves

Compare each TabICL context against XGBoost trained on the same splits.

In [ ]:
if RUN_TEST_COMPARISON:
    for reference, names in [
        ('xgboost', ['xgboost', 'tabicl']),
        ('xgboost_train_val', ['xgboost_train_val', 'tabicl_train_val'])
    ]:
        group = [p for (_, name), p in comparison.predictions.items() if name in names]
        display(bootstrap_by_day(group, baby_days('test'), reference=reference, n_boot=1000).round(3))
        for label in LABELS:
            plot_performance([p for p in group if p.label == label], title=f'{label}: {reference} context comparison')

## 7. Per-baby results and probability consistency

Aggregate scores can hide weaker performance on Imogen, the intended future user. Small per-baby samples make estimates noisy. We skip groups without both outcome classes because ROC-AUC is undefined.

The real events are nested: sleep within 30 minutes implies sleep within 60. Two independently fitted models can violate that probability ordering. We report violations without silently altering predictions or benchmark scores.

In [ ]:
if RUN_TEST_COMPARISON:
    by_baby = []
    for pred in comparison.predictions.values():
        for baby in test.baby_name.dropna().unique():
            mask = (test.baby_name == baby).to_numpy()
            if np.unique(pred.y_true[mask]).size < 2:
                print('Skipping single-class group:', baby, pred.label)
                continue
            sub = pred.model_copy(update={'y_true': pred.y_true[mask], 'p': pred.p[mask]})
            by_baby.append(metrics_table([sub]).reset_index().assign(baby=str(baby)))
    if by_baby:
        display(pd.concat(by_baby).set_index(['baby', 'label', 'model']).sort_index().round(3))
    for name in ['tabicl', 'tabicl_train_val']:
        p30 = comparison.predictions[(LABELS[0], name)].p
        p60 = comparison.predictions[(LABELS[1], name)].p
        print(name, 'fraction with P(30) > P(60):', float((p30 > p60).mean()))

## 8. Feature explanations and what to conclude

The existing SHAP utilities are TreeSHAP: they explain the trees, not TabICL. The optional cell below keeps that analysis available, clearly separated. Do not interpret it as TabICL's reasoning. Explaining the transformer requires a separate model-agnostic analysis and many extra inference calls.

After running, compare TabICL against each baseline on both horizons: ranking, probability error, baby-day gap intervals and Imogen's results. Check whether newer context helps more than model choice. Do not claim a winner from unexecuted cells or choose settings repeatedly on test. Choose ensemble size/context on validation and confirm any improvement on newly collected future data.

In [ ]:
RUN_TREE_SHAP = False  # can be expensive, especially random forest
if RUN_TREE_COMPARISON and RUN_TREE_SHAP:
    display(val.shap_importance())
    val.plot_shap_summaries()